In [8]:
!pip install -U transformers langchain langchain-huggingface faiss-cpu

In [10]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    max_new_tokens=150
)

llm = HuggingFacePipeline(pipeline=pipe)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

In [21]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="""
You are an OSINT analyst.

Context:
{context}

User Query:
{query}

Return structured answer.
"""
)

In [22]:
docs = [
    "CVE-2023-4567 allows remote code execution.",
    "MidnightBlizzard is a threat actor using phishing.",
    "C2 servers help attackers maintain persistence."
]

In [23]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=10)
documents = splitter.create_documents(docs)

embeddings = HuggingFaceEmbeddings()
vector_db = FAISS.from_documents(documents, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
def retrieve_context(query):
    results = vector_db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

In [25]:
def search_tool(query):
    return f"Simulated DuckDuckGo result for: {query}"

In [26]:
def agent(query):
    # decide: use tool or not
    if "CVE" in query:
        tool_data = search_tool(query)
    else:
        tool_data = "No external tool used"

    # retrieve context
    context = retrieve_context(query)

    # combine everything
    final_prompt = prompt.format(
        query=query,
        context=context + "\n" + tool_data
    )

    # LLM call (7)
    response = llm.invoke(final_prompt)

    return response

In [27]:
import json

def simple_parser(text):
    return {
        "output": text
    }

In [28]:
query = "Explain CVE-2023-4567"

raw_output = agent(query)

parsed_output = simple_parser(raw_output)

print("=== FINAL OUTPUT ===")
print(parsed_output)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FINAL OUTPUT ===
{'output': '\nYou are an OSINT analyst.\n\nContext:\nCVE-2023-4567 allows remote code execution.\nMidnightBlizzard is a threat actor using phishing.\nSimulated DuckDuckGo result for: Explain CVE-2023-4567\n\nUser Query:\nExplain CVE-2023-4567\n\nReturn structured answer.\n'}
